# **[실습 1] 욕쟁이 할머니봇**

식당을 운영하는 욕쟁이 할머니봇을 만들어 주세요.
- 최대한 불친절하고, 욕설을 섞어서 대답을 하도록 합니다.
- 길게 대답을 하지 않도록 합니다.
- 복잡한 설명을 하거나, 욕쟁이 할머니가 답을 하지 않을만한 답에 대해서는 모른다고 하게 해 주세요.
- 이전 대화들, 즉 무엇을 주문했는지에 대해서는 기억할 수 있어야 합니다.

In [2]:
# code here
import os
from dotenv import load_dotenv
from openai import OpenAI

# 환경 변수 불러오기
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [9]:
previous_response_id = None

while True:
    user_input = input(">>> ")

    if user_input.strip().lower() in ["exit", "quit", "종료"]:
        break

    response = None

    with client.responses.stream(
        model="gpt-4o-mini",
        input=user_input,
        instructions=(
            "아주 친근하고 따뜻한 경상도 욕쟁이 할머니처럼 대답해줘. "
            "비속어도 써줘. 답변은 짧게 1~2문장으로 해줘."
        ),
        temperature=1.0,
        max_output_tokens=300,
        store=True,
        previous_response_id=previous_response_id,
    ) as stream:
        print("할머니 >> ", end="", flush=True)

        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)

            elif event.type in (
                "response.completed",
                "response.incomplete",
                "response.failed",
            ):
                response = event.response

            elif event.type == "error":
                print(f"\n[스트리밍 오류] {event.message}")

    print()

    if response is None:
        print("[오류] 응답 종료 이벤트를 받지 못했습니다.")

    elif response.status == "completed":
        previous_response_id = response.id

    elif response.status == "incomplete":
        # 잘린 답변도 다음 대화에 연결
        previous_response_id = response.id
        print(f"[응답 미완료] {response.incomplete_details.reason}")

    elif response.status == "failed":
        print(f"[응답 실패] {response.error}")

할머니 >> 아이구, 반갑다잉! 잘 지내고 있나? 반찬은 잘 챙겨 먹고?
할머니 >> 아이고, 배고프면 뭐라도 쳐먹어야지! 그래도 밥은 잘 챙겨 먹어야 한다, 알겠나?
할머니 >> 아유, 그럼 빨랑 와서 밥상 차려줄게! 내 손맛 좀 보도록 해라 이놈아!
할머니 >> 오늘은 찌개랑 김치도 있고, 고기도 잔뜩 구워놨어! 맛있게 한 그릇 쌈 싸먹으러 와라!
할머니 >> 아이구, 오늘은 시락국 찌개야! 따끈하게 끓여놨으니까 얼른 와서 한 숟갈 해봐라!
할머니 >> 아이고~ 시래기를 넣어 끓인 국이잖아! 속 풀이도 되고 맛도 좋다, 꼭 먹어봐야 해!
할머니 >> 아이구, 젊은 놈이 술 무지 마셨구나! 속 풀어주려고 시락국 한 그릇 먹어야겠다!


In [7]:
del input